# 03 Train Baseline NCF

The baseline Neural Collaborative Filtering model uses only `user_id` and `movie_id` embeddings. It is the reference point for all context-aware variants.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.preprocessing import build_metadata_from_mappings
from src.train import train_model
from src.evaluate import evaluate_ranking
from src.analysis import append_or_replace_metrics
from src.utils import load_json, ensure_dir, set_seed

set_seed()
ensure_dir(config.RESULTS_DIR)
ensure_dir(config.MODELS_DIR)

In [ ]:
train_df = pd.read_csv(config.PROCESSED_DATA_DIR / 'train.csv')
val_df = pd.read_csv(config.PROCESSED_DATA_DIR / 'validation.csv')
val_candidates = pd.read_csv(config.PROCESSED_DATA_DIR / 'val_candidates.csv')
mappings = load_json(config.PROCESSED_DATA_DIR / 'mappings.json')
metadata = build_metadata_from_mappings(mappings)
print(train_df.shape, val_df.shape, val_candidates.shape)

## Train

The model is optimized with `BCEWithLogitsLoss` using sampled positives and negatives.

In [ ]:
model, history = train_model('baseline', train_df, val_df, metadata, config.MODEL_SAVE_PATHS['baseline'])
display(history.tail())

In [ ]:
history.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', title='Baseline NCF Loss')
plt.ylabel('BCE loss')
plt.tight_layout()
plt.show()

## Validation Ranking Evaluation

Ranking quality is measured on the fixed validation candidate set using HR@10, NDCG@10, and Recall@10.

In [ ]:
metrics, per_group, scored = evaluate_ranking(model, val_candidates, metadata, 'baseline', k=config.TOP_K)
row = {'split': 'validation', 'model': 'baseline', 'used_features': 'user_id + movie_id', **metrics}
metrics_df = append_or_replace_metrics(config.RESULTS_DIR / 'metrics.csv', [row])
display(pd.DataFrame([row]))
display(metrics_df)